# 📄 Notebook 2: Comment Ordering & Pagination

When you join a live video that's been running for 30 minutes, you want to see the most recent comments first — and then scroll up to read older ones. But with thousands of comments pouring in, how do you load them efficiently without crashing the database?

This notebook explores **offset vs cursor pagination**, shows why ordering matters in fast-moving feeds, and builds the pagination logic you'd describe in a system design interview.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why you can't just load all comments at once
- How offset pagination works (and why it breaks in live feeds)
- How cursor pagination works (and why it's the right choice)
- How to implement infinite scroll with cursor-based pagination
- The performance difference between offset and cursor approaches

## 🛠️ Setup

### 1. Start the infrastructure

Open a terminal and run:

```bash
cd system-designs/fb-live-comments
docker-compose up -d
```

This starts:
- **PostgreSQL** on `localhost:5432` (database: `live_comments`, user: `demo`, password: `demo`)
- **Redis** on `localhost:6379`
- **Adminer** at [http://localhost:8080](http://localhost:8080) — visual database browser
- **RedisInsight** at [http://localhost:5540](http://localhost:5540) — visual Redis browser

### 2. Select the Jupyter kernel

In VS Code, click the kernel picker (top-right of the notebook) and select the `.venv` kernel from this lab folder. If it doesn't appear, reload the VS Code window (`Cmd+Shift+P` → "Reload Window").

If you haven't created the virtual environment yet:

```bash
cd system-designs/fb-live-comments
uv venv
source .venv/bin/activate
uv sync
```

In [ ]:
import psycopg2
import time

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "live_comments",
    "user": "demo",
    "password": "demo",
}


def get_db_connection():
    return psycopg2.connect(**DB_CONFIG)


# Test connection
try:
    conn = get_db_connection()
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM comments")
    count = cur.fetchone()[0]
    cur.execute("SELECT COUNT(*) FROM live_videos WHERE is_live = TRUE")
    live_count = cur.fetchone()[0]
    conn.close()
    print(f"✅ Connected to PostgreSQL")
    print(f"   📊 {count} comments across {live_count} live videos")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")
    print("   Run: cd system-designs/fb-live-comments && docker-compose up -d")

## 🤔 Why Pagination Matters

Right now, Video 1 has about ~250 comments. That's manageable. But in production, a viral live video (think a presidential debate or a big sports game) can accumulate **millions** of comments.

Loading ALL comments at once would:

1. **Take forever to transfer** — sending millions of rows over the network is slow
2. **Consume tons of memory** — the client (phone/browser) can't hold millions of comments in memory
3. **Overwhelm the database** — scanning and returning huge result sets blocks other queries

The solution? **Pagination** — load comments in small pages (10–20 at a time), just like infinite scroll in a mobile app.

But *how* you paginate matters a lot. Let's explore two approaches.

In [ ]:
# Show the problem: loading ALL comments at once

conn = get_db_connection()
cur = conn.cursor()

start = time.time()
cur.execute("""
    SELECT c.id, u.display_name, c.message, c.created_at
    FROM comments c JOIN users u ON c.user_id = u.id
    WHERE c.live_video_id = 1
    ORDER BY c.id DESC
""")
all_comments = cur.fetchall()
elapsed = (time.time() - start) * 1000

print(f"📦 Loaded ALL {len(all_comments)} comments in {elapsed:.1f}ms")
print(f"   That's fine for {len(all_comments)} comments...")
print(f"   But imagine loading 10 million comments at once! 🤯")
print()
print("First 5 comments (newest first):")
for comment in all_comments[:5]:
    print(f"   #{comment[0]} [{comment[1]}]: {comment[2][:40]}...")
conn.close()

## 📏 Approach 1: Offset Pagination

The simplest way to paginate is **offset pagination**. Here's how it works:

- **Page 1**: `SELECT ... LIMIT 10 OFFSET 0` (rows 1–10)
- **Page 2**: `SELECT ... LIMIT 10 OFFSET 10` (rows 11–20)
- **Page 3**: `SELECT ... LIMIT 10 OFFSET 20` (rows 21–30)

The API looks like:
```
GET /comments/1?offset=0&pageSize=10
GET /comments/1?offset=10&pageSize=10
GET /comments/1?offset=20&pageSize=10
```

Simple to understand. Simple to implement. But it has serious problems in live feeds...

In [ ]:
def fetch_comments_offset(video_id, offset, page_size=10):
    """Fetch comments using offset pagination."""
    conn = get_db_connection()
    cur = conn.cursor()

    start = time.time()
    cur.execute(
        """
        SELECT c.id, u.display_name, c.message, c.created_at
        FROM comments c JOIN users u ON c.user_id = u.id
        WHERE c.live_video_id = %s
        ORDER BY c.id DESC
        LIMIT %s OFFSET %s
    """,
        (video_id, page_size, offset),
    )

    comments = cur.fetchall()
    elapsed = (time.time() - start) * 1000
    conn.close()

    return comments, elapsed


# Simulate scrolling through pages
print("📜 Offset Pagination — Scrolling through Video 1")
print("=" * 60)

for page in range(5):
    offset = page * 10
    comments, ms = fetch_comments_offset(1, offset, 10)
    print(f"\nPage {page + 1} (OFFSET {offset})  [{ms:.1f}ms]")
    for c in comments[:3]:  # Show first 3 of each page
        print(f"   #{c[0]} [{c[1]}]: {c[2][:40]}")
    if len(comments) > 3:
        print(f"   ... and {len(comments) - 3} more")

### ⚠️ The Problem with Offset Pagination in Live Feeds

Offset pagination has **two critical problems** in a live comment feed:

#### Problem 1: Performance Degrades with Depth

When you say `OFFSET 1000000`, the database doesn't magically jump to row 1,000,000. It has to **scan through all 1,000,000 rows** and throw them away, then return the next 10. The deeper you paginate, the slower it gets.

#### Problem 2: Unstable Results (The Sliding Window Problem)

This is the killer. In a live feed, new comments are constantly being inserted. Watch what happens:

```
Time 1: User loads Page 1 (OFFSET 0, LIMIT 10)
        → Gets comments 500, 499, 498, 497, 496, 495, 494, 493, 492, 491

Time 2: 3 new comments posted (503, 502, 501)

Time 3: User scrolls to Page 2 (OFFSET 10, LIMIT 10)
        → The 10 newest are now: 503, 502, 501, 500, 499, 498, 497, 496, 495, 494
        → OFFSET 10 skips those, returning: 493, 492, 491, 490, 489...
        → User sees 493, 492, 491 AGAIN! (duplicates from page 1)
```

The offset shifted because new rows were inserted at the top. The user sees duplicates and misses comments. Let's prove it.

In [ ]:
# Demonstrate: insert new comments while paginating with offset

conn = get_db_connection()
cur = conn.cursor()

# Page 1: get the 10 newest comments
cur.execute("""
    SELECT c.id, c.message FROM comments c
    WHERE c.live_video_id = 1 ORDER BY c.id DESC LIMIT 10 OFFSET 0
""")
page1 = cur.fetchall()
page1_ids = {c[0] for c in page1}
print("Page 1 (before new comments):")
print(f"   Comment IDs: {[c[0] for c in page1]}")

# Simulate: 3 new comments arrive while user is reading page 1
for msg in ["New comment A!", "New comment B!", "New comment C!"]:
    cur.execute(
        "INSERT INTO comments (live_video_id, user_id, message) VALUES (1, 1, %s) RETURNING id",
        (msg,),
    )
conn.commit()

# Page 2: user scrolls down with OFFSET 10
cur.execute("""
    SELECT c.id, c.message FROM comments c
    WHERE c.live_video_id = 1 ORDER BY c.id DESC LIMIT 10 OFFSET 10
""")
page2 = cur.fetchall()
page2_ids = {c[0] for c in page2}

duplicates = page1_ids & page2_ids
print(f"\nPage 2 (after 3 new comments were inserted):")
print(f"   Comment IDs: {[c[0] for c in page2]}")
print(f"\n⚠️  Duplicates between pages: {duplicates}")
print(f"   The user would see {len(duplicates)} comment(s) TWICE!")

# Clean up the test comments
cur.execute(
    "DELETE FROM comments WHERE message IN ('New comment A!', 'New comment B!', 'New comment C!')"
)
conn.commit()
conn.close()

## 🎯 Approach 2: Cursor Pagination (The Right Way)

Instead of saying "skip N rows", cursor pagination says **"give me rows after this specific point."**

The **cursor** is the ID of the last comment the user saw. Since comment IDs are auto-incrementing (`BIGSERIAL`), they're naturally ordered.

Here's how it works:

- **Page 1**: `SELECT ... WHERE video_id = 1 ORDER BY id DESC LIMIT 10`
  - Returns comments 500–491. Last seen ID: **491**
- **Page 2**: `SELECT ... WHERE video_id = 1 AND id < 491 ORDER BY id DESC LIMIT 10`
  - Returns comments 490–481. Last seen ID: **481**
- **Page 3**: `SELECT ... WHERE video_id = 1 AND id < 481 ORDER BY id DESC LIMIT 10`
  - Returns comments 480–471.

The API looks like:
```
GET /comments/1?pageSize=10              → first page
GET /comments/1?cursor=491&pageSize=10   → next page
GET /comments/1?cursor=481&pageSize=10   → next page
```

### Why is this better?

1. **Stable**: New comments get IDs > 500, so `WHERE id < 491` always returns the same rows
2. **Fast**: The database uses the index on `(live_video_id, id)` to jump directly to the cursor — no scanning
3. **No duplicates**: The cursor is a fixed reference point, unaffected by new inserts

In [ ]:
def fetch_comments_cursor(video_id, cursor=None, page_size=10):
    """Fetch comments using cursor pagination.

    cursor: the ID of the last comment seen (fetch comments OLDER than this)
    If cursor is None, fetch the newest comments.
    """
    conn = get_db_connection()
    cur = conn.cursor()

    start = time.time()
    if cursor is None:
        cur.execute(
            """
            SELECT c.id, u.display_name, c.message, c.created_at
            FROM comments c JOIN users u ON c.user_id = u.id
            WHERE c.live_video_id = %s
            ORDER BY c.id DESC
            LIMIT %s
        """,
            (video_id, page_size),
        )
    else:
        cur.execute(
            """
            SELECT c.id, u.display_name, c.message, c.created_at
            FROM comments c JOIN users u ON c.user_id = u.id
            WHERE c.live_video_id = %s AND c.id < %s
            ORDER BY c.id DESC
            LIMIT %s
        """,
            (video_id, cursor, page_size),
        )

    comments = cur.fetchall()
    elapsed = (time.time() - start) * 1000
    conn.close()

    next_cursor = comments[-1][0] if comments else None
    return comments, next_cursor, elapsed


# Simulate scrolling with cursor pagination
print("📜 Cursor Pagination — Scrolling through Video 1")
print("=" * 60)

cursor = None
for page in range(5):
    comments, next_cursor, ms = fetch_comments_cursor(1, cursor, 10)
    print(f"\nPage {page + 1} (cursor={cursor})  [{ms:.1f}ms]")
    for c in comments[:3]:
        print(f"   #{c[0]} [{c[1]}]: {c[2][:40]}")
    if len(comments) > 3:
        print(f"   ... and {len(comments) - 3} more")
    cursor = next_cursor
    print(f"   → Next cursor: {cursor}")

In [ ]:
# Prove: cursor pagination is STABLE even when new comments arrive

conn = get_db_connection()
cur = conn.cursor()

# Page 1: get the 10 newest comments
comments_p1, next_cursor, _ = fetch_comments_cursor(1, cursor=None, page_size=10)
page1_ids = [c[0] for c in comments_p1]
print("Page 1 (before new comments):")
print(f"   Comment IDs: {page1_ids}")
print(f"   Next cursor: {next_cursor}")

# Simulate: 3 new comments arrive
for msg in ["Cursor test A!", "Cursor test B!", "Cursor test C!"]:
    cur.execute(
        "INSERT INTO comments (live_video_id, user_id, message) VALUES (1, 1, %s)",
        (msg,),
    )
conn.commit()

# Page 2: use the cursor from page 1
comments_p2, _, _ = fetch_comments_cursor(1, cursor=next_cursor, page_size=10)
page2_ids = [c[0] for c in comments_p2]

duplicates = set(page1_ids) & set(page2_ids)
print(f"\nPage 2 (after 3 new comments were inserted):")
print(f"   Comment IDs: {page2_ids}")
print(f"\n✅ Duplicates between pages: {duplicates}")
print("   ZERO duplicates! Cursor pagination is stable! 🎉")

# Clean up
cur.execute(
    "DELETE FROM comments WHERE message IN ('Cursor test A!', 'Cursor test B!', 'Cursor test C!')"
)
conn.commit()
conn.close()

## ⚡ Performance: Offset vs Cursor at Scale

With a small dataset, both approaches feel instant. The real difference shows up when you paginate **deep** into a large dataset. Let's generate more data and compare.

We'll insert 10,000 extra comments and then measure how long it takes to fetch page N using each approach.

In [ ]:
# Insert many comments to see the performance difference at scale
conn = get_db_connection()
cur = conn.cursor()

# Add 10,000 more comments for performance testing
print("📊 Inserting 10,000 test comments for performance comparison...")
cur.execute("""
    INSERT INTO comments (live_video_id, user_id, message, created_at)
    SELECT 1, (floor(random() * 50) + 1)::int, 'Performance test comment ' || i,
           NOW() - interval '1 second' * i
    FROM generate_series(1, 10000) AS i
""")
conn.commit()

cur.execute("SELECT COUNT(*) FROM comments WHERE live_video_id = 1")
total = cur.fetchone()[0]
print(f"   Total comments for video 1: {total}")

# Compare at various "depths"
print("\n📊 Offset vs Cursor — Performance at Different Depths")
print("=" * 65)
print(f"{'Depth':<12} {'Offset (ms)':<15} {'Cursor (ms)':<15} {'Winner'}")
print("-" * 65)

for depth in [10, 100, 1000, 5000, 9000]:
    # Offset: get page at this depth
    start = time.time()
    cur.execute(
        """
        SELECT c.id, c.message FROM comments c
        WHERE c.live_video_id = 1 ORDER BY c.id DESC LIMIT 10 OFFSET %s
    """,
        (depth,),
    )
    cur.fetchall()
    offset_ms = (time.time() - start) * 1000

    # Get the cursor value at this depth for fair comparison
    cur.execute(
        """
        SELECT c.id FROM comments c
        WHERE c.live_video_id = 1 ORDER BY c.id DESC LIMIT 1 OFFSET %s
    """,
        (depth,),
    )
    cursor_val = cur.fetchone()[0]

    # Cursor: get page using cursor
    start = time.time()
    cur.execute(
        """
        SELECT c.id, c.message FROM comments c
        WHERE c.live_video_id = 1 AND c.id < %s ORDER BY c.id DESC LIMIT 10
    """,
        (cursor_val,),
    )
    cur.fetchall()
    cursor_ms = (time.time() - start) * 1000

    winner = "Cursor ✅" if cursor_ms < offset_ms else "Offset"
    print(f"{depth:<12} {offset_ms:<15.2f} {cursor_ms:<15.2f} {winner}")

# Clean up: remove the test comments
cur.execute("DELETE FROM comments WHERE message LIKE 'Performance test comment%'")
conn.commit()
conn.close()

print("\n💡 Cursor pagination has consistent performance regardless of depth!")
print("   Offset gets slower and slower as you paginate deeper.")

## 🔄 Building the Complete Pagination API

Now let's put it all together into a complete pagination function that returns a proper API response. This is what you'd build in a real backend:

```
GET /comments/{video_id}?cursor={id}&pageSize=10&sort=desc
```

The response includes:
- `comments[]` — the page of comments
- `next_cursor` — the cursor to pass for the next page (or `null` if no more pages)
- `has_more` — boolean indicating if there are more pages

The trick to knowing if there are more pages: **fetch `pageSize + 1` rows**, and if you get the extra row, there are more pages. Return only `pageSize` rows to the client.

In [ ]:
import json


def paginate_comments(video_id, cursor=None, page_size=10, sort="desc"):
    """Complete pagination function that returns API-style response."""
    conn = get_db_connection()
    cur = conn.cursor()

    order = "DESC" if sort == "desc" else "ASC"
    op = "<" if sort == "desc" else ">"

    if cursor is None:
        cur.execute(
            f"""
            SELECT c.id, u.display_name, c.message, c.created_at
            FROM comments c JOIN users u ON c.user_id = u.id
            WHERE c.live_video_id = %s
            ORDER BY c.id {order}
            LIMIT %s
        """,
            (video_id, page_size + 1),
        )  # +1 to check if more pages exist
    else:
        cur.execute(
            f"""
            SELECT c.id, u.display_name, c.message, c.created_at
            FROM comments c JOIN users u ON c.user_id = u.id
            WHERE c.live_video_id = %s AND c.id {op} %s
            ORDER BY c.id {order}
            LIMIT %s
        """,
            (video_id, cursor, page_size + 1),
        )

    rows = cur.fetchall()
    conn.close()

    has_more = len(rows) > page_size
    comments = rows[:page_size]

    return {
        "comments": [
            {
                "id": r[0],
                "user": r[1],
                "message": r[2],
                "created_at": str(r[3]),
            }
            for r in comments
        ],
        "next_cursor": comments[-1][0] if comments and has_more else None,
        "has_more": has_more,
        "page_size": page_size,
    }


# Demo: simulate infinite scroll
print("📱 Simulating Infinite Scroll (newest → oldest)")
print("=" * 55)

cursor = None
total_loaded = 0
for scroll in range(3):
    result = paginate_comments(1, cursor=cursor, page_size=5)
    total_loaded += len(result["comments"])

    print(f"\n{'─' * 55}")
    print(f"Scroll #{scroll + 1}  |  Cursor: {cursor}  |  Has more: {result['has_more']}")
    print(f"{'─' * 55}")
    for c in result["comments"]:
        print(f"  💬 {c['user']}: {c['message'][:35]}")

    cursor = result["next_cursor"]

print(f"\n📊 Loaded {total_loaded} comments in {scroll + 1} scrolls")
print("   Each scroll only fetched 5 comments — efficient! ✅")

## 🧭 Ordering: Newest-First vs Oldest-First

Live comment feeds typically show the **newest comments at the bottom** (like a chat app). But when you're loading comments, you need to think about ordering:

| Scenario | Order | Why |
|----------|-------|-----|
| Initial page load | Newest first (`DESC`) | Show what just happened |
| Scroll up for history | Older comments (`DESC` with cursor) | Load older pages |
| Read from beginning | Oldest first (`ASC`) | Chronological order |

Our `paginate_comments` function supports both via the `sort` parameter. Let's see the difference.

In [ ]:
print("📊 Same comments, different ordering:")
print()

# Newest first (for initial page load)
result_desc = paginate_comments(1, page_size=5, sort="desc")
print("⬇️  Newest First (sort=desc) — for 'what just happened?'")
for c in result_desc["comments"]:
    print(f"   #{c['id']} {c['user']}: {c['message'][:35]}")

print()

# Oldest first (for reading in order)
result_asc = paginate_comments(1, page_size=5, sort="asc")
print("⬆️  Oldest First (sort=asc) — for 'read from the beginning'")
for c in result_asc["comments"]:
    print(f"   #{c['id']} {c['user']}: {c['message'][:35]}")

print()
print("💡 In a live comment feed:")
print("   • Initial load: newest-first (show the latest comments)")
print("   • Scroll up: fetch older comments with cursor")
print("   • New comments: append at the bottom via SSE (see Notebook 1)")

## 📊 Offset vs Cursor: Summary Comparison

| Feature | Offset Pagination | Cursor Pagination |
|---------|-------------------|-------------------|
| **Implementation** | Simple (`LIMIT/OFFSET`) | Slightly more complex (`WHERE id < cursor`) |
| **Performance** | Degrades at large offsets | Constant regardless of depth |
| **Stability** | Breaks when new items are inserted | Always stable |
| **Jump to page** | Easy (calculate offset) | Not possible (must traverse sequentially) |
| **Best for** | Static data, small datasets | Live feeds, large datasets |
| **Used by** | Simple admin panels | Facebook, Twitter, Instagram feeds |

## 📚 Summary

### Key Takeaways

1. **Never load all comments at once** — pagination is essential for performance and UX
2. **Offset pagination is simple but broken for live feeds** — new inserts cause duplicates and skipped items
3. **Cursor pagination is stable and fast** — performance doesn't degrade with depth
4. **Use BIGSERIAL IDs as cursors** — they're monotonically increasing, indexed, and unique
5. **Return `has_more` and `next_cursor`** — the client needs to know when to stop scrolling

### How This Maps to the System Design Interview

| Concept | What to Say |
|---------|-------------|
| Pagination choice | "I'd use cursor-based pagination because offset is unstable in a fast-moving feed" |
| Cursor field | "The comment's auto-incrementing ID is a natural cursor — it's indexed and monotonic" |
| Page size | "10–20 comments per page balances UX responsiveness with network efficiency" |
| Ordering | "Newest-first for initial load, with infinite scroll loading older pages" |

### Next Up

In **Notebook 3**, we'll tackle **scaling live comments** — what happens when millions of viewers watch the same video and thousands of comments pour in per second.